# Center-bearing and center-distance plots

Choose a saved recording, probe and phase. Like `spatial_cell_plotting.ipynb`, this notebook exports PNG/SVG unit figures and previews a selected unit. It reads saved analysis only; no tracking, spikes, shuffles, classifications or theta calculations are rerun. Run on `hhw9l84` with the `rfmapping` kernel.


In [ ]:
%matplotlib inline
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import display

from EBB.center_cell_io import load_center_analysis
from EBB.center_cell_plotting import plot_center_unit, plot_center_tuning, plot_center_population
from Utils.plotting import apply_light_plot_style

apply_light_plot_style()


In [ ]:
recording_root = Path("/mnt/senzailab/Kai/#Recording/m20")
date = "260918"
recording_number = 11
probe_name = "A"
phase_key = "baseline"


session_id = f"{recording_root.name}_{date}_{recording_number}"
session_dir = recording_root / date / f"{date}_{recording_number}"
result_dir = session_dir / "data" / "center_cells" / f"Probe{probe_name}" / phase_key
plot_directory = result_dir / "figures"
unit_id = 194
is_save = True
center_results = load_center_analysis(result_dir)
display(center_results["summary"])


## Export all saved units and preview one


In [ ]:
if is_save:
    unit_directory = plot_directory / "all_units"
    unit_directory.mkdir(parents=True, exist_ok=True)
    for saved_unit in center_results["summary"].index:
        figure = plot_center_unit(center_results, saved_unit, title=f"{session_id} | Unit {saved_unit}")
        for suffix in ("png", "svg"):
            figure.savefig(unit_directory / f"unit_{saved_unit}.{suffix}", dpi=200, facecolor="white", transparent=False)
        plt.close(figure)
    print(f"Saved unit figures: {unit_directory}")

figure = plot_center_unit(center_results, unit_id, title=f"{session_id} | Unit {unit_id}")
display(figure)
plt.close(figure)


## Selected tuning curve

CB supports `line`, `heatmap`, `polar` and `donut`, using clockwise egocentric angles (0° forward). CD supports `line` and `heatmap` with the native centimeter axis. Gray/missing bins remain distinct from measured zero responses. The classification comes from the saved analysis.


In [ ]:
feature = "cb"  # "cb" or "cd"
plot_style = "polar"  # Use "line" or "heatmap" for CD.
figure = plot_center_tuning(
    center_results, unit_id, feature=feature, plot_style=plot_style,
    title=f"{session_id} | Unit {unit_id}",
)
if is_save:
    plot_directory.mkdir(parents=True, exist_ok=True)
    for suffix in ("png", "svg"):
        figure.savefig(plot_directory / f"unit_{unit_id}_{feature}_{plot_style}.{suffix}", dpi=200, facecolor="white", transparent=False)
display(figure)
plt.close(figure)


## Population tuning

Choose `all`, `positive` or `unassessed` independently for each feature. Rows retain saved unit order; values remain Hz on the native angular/distance bins. Zero and unvisited bins retain their different meanings.


In [ ]:
cell_selection = "all"
for feature in ("cb", "cd"):
    labels = center_results["summary"][f"{feature}_is_cell"]
    if cell_selection == "positive":
        selected_units = labels.index[labels.fillna(False)].tolist()
    elif cell_selection == "unassessed":
        selected_units = labels.index[labels.isna()].tolist()
    elif cell_selection == "all":
        selected_units = labels.index.tolist()
    else:
        raise ValueError("cell_selection must be 'all', 'positive' or 'unassessed'")
    figure = plot_center_population(center_results, feature=feature, unit_ids=selected_units)
    if is_save:
        for suffix in ("png", "svg"):
            figure.savefig(plot_directory / f"{feature}_{cell_selection}_heatmap.{suffix}", dpi=200, facecolor="white", transparent=False)
    display(figure)
    plt.close(figure)
